In [1]:
from robot import Robot, MotomanRobot
from planning_scene import PlanningScene
from geometric_trajopt_nlopt import PoseTrajOpt
import numpy as np
import scipy as sp
import transformations as tf
import nlopt

In [2]:
# test the robot with the scene
# add environment collisions
pcd_total = []
# shelf-bottom
num_points = 5000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 5000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 5000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 5000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 5000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


torso_b1 = ["torso_joint_b1"]
left = [
    "arm_left_joint_1_s",
    "arm_left_joint_2_l",
    "arm_left_joint_3_e",
    "arm_left_joint_4_u",
    "arm_left_joint_5_r",
    "arm_left_joint_6_b",
    "arm_left_joint_7_t",
]
right = [
    "arm_right_joint_1_s",
    "arm_right_joint_2_l",
    "arm_right_joint_3_e",
    "arm_right_joint_4_u",
    "arm_right_joint_5_r",
    "arm_right_joint_6_b",
    "arm_right_joint_7_t",
]
robot_joint_names = right#torso_b1 + left + right
robot = MotomanRobot(selected_joint_names=robot_joint_names)
# scene = PlanningScene(robot, scene_pcd=pcd_total)
scene = PlanningScene(robot, scene_pcd=None, octree_res=0.02)
scene.update_scene_pcd(pcd_total)


In [3]:
q_start = np.array([0.0, 0, 0, 0, 0, 0, 0])
robot.set_selected_joint_values(q_start)
scene.visualize()

[TriangleMesh with 194 points and 384 triangles.,
 TriangleMesh with 579 points and 1154 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 732 points and 1460 triangles.,
 TriangleMesh with 803 points and 1590 triangles.,
 TriangleMesh with 451 points and 898 triangles.,
 TriangleMesh with 694 points and 1384 triangles.,
 TriangleMesh with 527 points and 1050 triangles.,
 TriangleMesh with 464 points and 924 triangles.,
 TriangleMesh with 225 points and 446 triangles.,
 TriangleMesh with 10905 points and 21834 triangles.,
 TriangleMesh with 17156 points and 34248 triangles.,
 TriangleMesh with 672 points and 1340 triangles.,
 TriangleMesh with 888 points an

In [4]:
# pos = np.array([0.9096643361780983, -0.22, 1.2246445275392399])
# quat = np.array([0.99997882565292, 0.0020695812292800746, -0.005795312726922581, 0.0021164663332217232])
# pose = tf.quaternion_matrix(quat)
# pose[:3,3] = pos

# * make sure the sampled joint is collision free
while True:
    # * generate the start pose by randomly sampling joint angles
    q_target = np.random.uniform(robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1])
    print('q_target:', q_target)
    # * visualize the start pose
    robot.set_selected_joint_values(q_target)
    # * check if the start pose is collision free
    col_list = scene.compute_collision_total()
    scene.visualize()
    # check for collision
    for link1, link2, obj1, obj2, col_result in col_list:
        print('collision:')
        print('link1:', link1)
        print('link2:', link2)
        print('result: ', col_result.isCollision())
        print('distance: ', -col_result.getContact(0).penetration_depth)
    if len(col_list) == 0:
        break
robot.set_selected_joint_values(q_target)
scene.visualize();

q_target: [-2.05712876 -1.25355314 -0.93964351 -0.90672781  1.33674832  0.72399279
 -0.30739746]


In [5]:
# obtain the goal pose
target_link = "arm_right_link_7_t"
robot.set_selected_joint_values(q_target)
target_pose = robot.get_link_pose(target_link)
print("target_pose: ", target_pose)

target_pose:  [[ 0.86460465 -0.48718654  0.12291489  0.15070965]
 [-0.03291465 -0.29902188 -0.95367843 -0.75327289]
 [ 0.50137354  0.8205091  -0.27457128  0.60785854]
 [ 0.          0.          0.          1.        ]]


In [6]:
distance = np.linalg.norm(q_target-q_start)
# n_waypoints = int(np.ceil(distance / (2*np.pi/180)))
n_waypoints = 10
print('distsance: ', distance)
print('n_waypoints: ', n_waypoints)
dist = distance / 5
solver = PoseTrajOpt(robot, scene, start_q=q_start, target_link=target_link, target_pose=target_pose, n_waypoints=n_waypoints, max_dist=dist)

distsance:  3.1486210142003714
n_waypoints:  10


In [7]:
import open3d as o3d
# initialization of the trajectory
q_init = np.zeros((n_waypoints, len(robot.selected_joint_dofids)))
q_init = np.linspace(q_start, q_target, n_waypoints+1)[1:]  # optimization variable: starting from the second point
# add purturbation to the initialization
q_init += np.random.normal(0, 10*np.pi/180, q_init.shape)
# clip the joint values
np.clip(q_init, robot.selected_joint_limits[:,0], robot.selected_joint_limits[:,1], out=q_init)
# visualize the initalization
vis = []
for i in range(n_waypoints):
    robot.set_selected_joint_values(q_init[i])
    vis_i = scene.visualize(False)
    vis += vis_i
o3d.visualization.draw_geometries(vis)


In [8]:
vis = []
i = 5
robot.set_selected_joint_values(q_init[i])
print('i: ', i)
scene.visualize(True);
# check collision at the point
collision_results = scene.compute_collision_total(security_margin=0.05, full=True)

for col_i in range(len(collision_results)):
    link1, link2, obj_idx1, obj_idx2, collision_result = collision_results[col_i]
    if link2 == "scene":
        n_contacts = collision_result.numContacts()
        for contact_i in range(n_contacts):
            contact = collision_result.getContact(contact_i)
            dist += (-contact.penetration_depth)
            print("scene contact distance: ", -contact.penetration_depth)
            normal = np.array(contact.normal)  # normal: p2 - p1
            pt = np.array(contact.pos)


# collision_results = scene.compute_collision_total(security_margin=0.001, full=True)
# # print('compute collision time: ', time.time()-start_time)

# # start_time = time.time()  # time for computing jacobian
# # if not in collision, then return the min distance
# for col_i in range(len(collision_results)):
#     link1, link2, obj_idx1, obj_idx2, collision_result = collision_results[col_i]
#     if link2 == "scene":
#         n_contacts = collision_result.numContacts()
#         for contact_i in range(n_contacts):
#             contact = collision_result.getContact(contact_i)
#             dist += (-contact.penetration_depth)
#             print("scene contact distance: ", -contact.penetration_depth)
#             normal = np.array(contact.normal)  # normal: p2 - p1
#             pt = np.array(contact.pos)



i:  5


In [9]:
# verify the collisoin on the initial trajectory
collision_constrs = solver.collision_constraints(q_init.flatten())
print('collision_constrs:', collision_constrs.min())

collision_constrs: 0.011799984842704338


In [10]:
opt = nlopt.opt(nlopt.LD_SLSQP, solver.n)
opt.set_min_objective(solver.objective_trajopt_f)
opt.set_lower_bounds(solver.lb)
opt.set_upper_bounds(solver.ub)
opt.add_inequality_mconstraint(solver.collision_constraint_nlopt_c, np.zeros(solver.constr_safety_margin.shape)+1e-8)
opt.add_inequality_mconstraint(solver.position_distance_constraint_nlopt_c, np.zeros(solver.constr_max_dist.shape)+1e-8)
# opt.add_inequality_constraint(solver.terminal_pose_trajopt_f, 1e-8)

opt.set_ftol_abs(1e-3)


In [11]:
xopt = opt.optimize(q_init.flatten())


waypoint distance objective:  6.161714040647045
terminal pose distance value:  0.5387789452909516
objective value:  11.54950349355656
collision violation: 
0.0
position distance constraints violation: 
0.607975588538901
waypoint distance objective:  24.07835052903553
terminal pose distance value:  1.9479195544190806
objective value:  43.557546073226334


/home/yinglong/Documents/research/task_motion_planning/non-prehensile-manipulation/motoman_ws/src/lab_vbnpm/playgrounds/optimal_control/planning_scene.py:150: RuntimeWarning: invalid value encountered in divide
  normal = normal / np.linalg.norm(normal)


collision violation: 
0.009089372505860518
position distance constraints violation: 
17.781108500634783
waypoint distance objective:  7.786827379816128
terminal pose distance value:  3.7433069006851976
objective value:  45.2198963866681
collision violation: 
0.03608213555478092
position distance constraints violation: 
1.9863566593571838
waypoint distance objective:  4.604849089271605
terminal pose distance value:  0.7808099120577108
objective value:  12.412948209848713
collision violation: 
0.0
position distance constraints violation: 
0.2741981219896087
waypoint distance objective:  5.287389679667472
terminal pose distance value:  0.2607514093523873
objective value:  7.894903773191345
collision violation: 
0.0
position distance constraints violation: 
0.20661015159879847
waypoint distance objective:  5.287389679667472
terminal pose distance value:  0.2607514093523873
objective value:  7.894903773191345
collision violation: 
0.0
position distance constraints violation: 
0.206610151598

In [12]:
minf = opt.last_optimum_value()
print("minimum value = ", minf)
print("result code = ", opt.last_optimize_result())
opt.get_errmsg()


minimum value =  3.152005667437595
result code =  3


In [13]:
# check the difference of x_opt and x_init
print('diff: ', np.linalg.norm(xopt-q_init.flatten()))

diff:  1.3737980658128846


In [14]:
# check constraint satisfaction
solver.compute_collision_constraints(xopt.flatten())
constrs = solver.collision_constraints(xopt)
print(np.linalg.norm(xopt.flatten() - solver.prev_qs))
print(np.all(constrs >= solver.safety_margin))
# check the violation
print(constrs[constrs<=solver.safety_margin])
# check the distance constraint
constrs = solver.position_distance_constraints(xopt)
print(np.all(constrs <= solver.max_dist))
# check the joint limits
constrs = solver.joint_limit_constraints(xopt)
print(np.all(constrs >= solver.cl_joint_limits.flatten()))
print(np.all(constrs <= solver.cu_joint_limits.flatten()))

0.0
True
[]
True
True
True


In [15]:
# visualize the result
vis = []
for i in range(n_waypoints):
    robot.set_selected_joint_values(q_init[i])
    vis_i = scene.visualize(False)
    vis += vis_i
for i in range(len(vis)):
    vis[i].paint_uniform_color([0,1,0])

vis_opt = []
for i in range(n_waypoints):
    robot.set_selected_joint_values(xopt.reshape(q_init.shape)[i])
    vis_i = scene.visualize(False)
    vis_opt += vis_i
for i in range(len(vis_opt)):
    vis_opt[i].paint_uniform_color([1,0,0])

o3d.visualization.draw_geometries(vis+vis_opt)

In [ ]:
def compute_collision_constraints(solver, qs):
    qs = np.array(qs).reshape(solver.n_waypoints, solver.ndof)
    robot = solver.robot
    scene = solver.scene
    cc_margin = 0.05
    constrs = []
    jacs = []  # jacobian shape: n_waypoints*n_cols x n_waypoints*ndof
    row = []
    col = []
    for i in range(solver.n_waypoints):
        # start_time = time.time()
        robot.set_selected_joint_values(qs[i])
        # handle self collision
        collision_results = scene.compute_collision_total(security_margin=cc_margin, full=True)
        # print('compute collision time: ', time.time()-start_time)

        # start_time = time.time()  # time for computing jacobian
        # if not in collision, then return the min distance
        for col_i in range(len(collision_results)):
            link1, link2, obj_idx1, obj_idx2, collision_result = collision_results[col_i]
            if link2 == "scene":
                n_contacts = collision_result.numContacts()
                for contact_i in range(n_contacts):
                    contact = collision_result.getContact(contact_i)
                    dist += (-contact.penetration_depth)
                    print("scene contact distance: ", -contact.penetration_depth)
                    normal = np.array(contact.normal)  # normal: p2 - p1
                    pt = np.array(contact.pos)
            # obtain the pose of the two links
            pose1 = robot.get_link_pose(link1)
            pose1_inv = np.linalg.inv(pose1)
            pose2 = np.eye(4)
            pose2_inv = np.eye(4)
            if link2 != "scene":
                pose2 = robot.get_link_pose(link2)
                pose2_inv = np.linalg.inv(pose2)
            if not collision_result.isCollision():
                dist = cc_margin
                # constrs.append(cc_margin)
                # continue  # need to compute jacobian as well
            else:
                dist = 0.0 # at least one contact. So we compute the mean of the contact distance
            # otherwise, compute jacobian. use the mean of each contact point
            # jac = np.zeros((robot.nv, 1))
            jac = np.zeros((len(robot.selected_joint_dofids)))
            n_contacts = collision_result.numContacts()
            for contact_i in range(n_contacts):
                contact = collision_result.getContact(contact_i)
                dist += (-contact.penetration_depth)
                print("contact distance: ", -contact.penetration_depth)
                # * compute jacobian
                # self collision:
                # d g(q) / d q = 1/g(q)*(J_p1(q) - J_p2(q))
                # NOTE: the point p1 and p2 are in the world frame
                # it seems that the contact point is the intermediate point between the two objects
                normal = np.array(contact.normal)  # normal: p2 - p1
                pt = np.array(contact.pos)
                p1 = pt - normal * 0.5 * (-contact.penetration_depth)
                p2 = pt + normal * 0.5 * (-contact.penetration_depth)
                # get the point on the link
                p1_local = pose1_inv[:3,:3]@p1 + pose1_inv[:3,3]
                p2_local = pose2_inv[:3,:3]@p2 + pose2_inv[:3,3]
                if link2 != "scene":
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac2 = robot.get_point_on_link_spatial_jacobian(link2, p2_local)
                    jac_i = np.tensordot(normal, jac2 - jac1, axes=1)
                    # jac_i = normal@(jac1 - jac2)
                else:
                    # collision with environment:
                    # d g(q) / d q = 1/g(q)*J_p1(q)
                    jac1 = robot.get_point_on_link_spatial_jacobian(link1, p1_local)
                    jac_i = np.tensordot(normal, -jac1, axes=1)
                    # jac_i = normal@jac1
                jac += jac_i

            if n_contacts > 0:
                dist = dist / n_contacts
                jac = jac / n_contacts
            constrs.append(dist)
            jacs.append(jac)  # shape: ndof
            row_i = np.zeros(jac.shape) + i*len(collision_results)+col_i # this is the index for the collisions
                                                                            # shape: ndof
            column_i = np.arange(len(jac)) + i*solver.ndof # this is the index for the joint angles
            row.append(row_i)
            col.append(column_i)
        # print('compute jacobian time: ', time.time()-start_time)
    constrs = np.array(constrs) # self.n_waypoints*len(collision_results)
    jacs_data = np.array(jacs).flatten()  # shape: n_waypoints*n_cols x ndof
    row = np.array(row).flatten()
    col = np.array(col).flatten()
    jacs = sp.sparse.coo_array((jacs_data, (row, col)), shape=(solver.n_waypoints*len(collision_results),
                                                                solver.n_waypoints*solver.ndof))


In [ ]:
compute_collision_constraints(solver, xopt.flatten())

contact distance:  0.04842599499477113
contact distance:  0.049636664422454964
contact distance:  0.04675741158688119
contact distance:  0.04692459319245536
contact distance:  0.027500517827492788
contact distance:  0.04861138677725592
contact distance:  0.04725893582627958
contact distance:  0.04967236395254366
contact distance:  0.04992885351479688
contact distance:  0.03900257411118459
contact distance:  0.049826667608107625
contact distance:  0.049973083436052876
contact distance:  0.04992885216768891
contact distance:  0.04842599499477113
contact distance:  0.04949555932491085
contact distance:  0.04975169983450486
contact distance:  0.04910007578038311
contact distance:  0.02750051782749254
contact distance:  0.04861138677725636
contact distance:  0.04725893582627962
contact distance:  0.049672363952543694
contact distance:  0.04992885351479662
contact distance:  0.039002574111184664
contact distance:  0.04982666760810782
contact distance:  0.049973083436052924
contact distance: 

In [ ]:
def naive_finite_diff(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # x is a numpy array of shape (n,)
    diff = eps
    grad = np.zeros(x.shape)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += diff
        x_minus = x.copy()
        x_minus[i] -= diff
        grad[i] = (func(x_plus) - func(x_minus)) / (2*diff)
    return grad

import time

def naive_finite_diff_jacobian(func, x, eps=1e-8):
    # given a function and the point for differentiation, compute the naive finite diff
    # func returns array of shape (M,)  #(M can be a list)
    # x is a numpy array of shape (n,)
    # result of shape (M,n)
    func_shape = list(func(x).shape)
    x_shape = list(x.shape)
    jacobian = np.zeros(list(func(x).shape)+list(x.shape)).reshape((np.prod(func_shape), np.prod(x_shape)))  # first flatten the func shape
    for i in range(np.prod(x_shape)):
        start_time = time.time()
        print('i: %d/%d' % (i, np.prod(x_shape)))
        x_plus = x.copy()
        x_plus.flat[i] += eps
        x_minus = x.copy()
        x_minus.flat[i] -= eps
        
        deriv = (func(x_plus) - func(x_minus)) / (2*eps)
        jacobian[:,i] = deriv.flatten()
        duration = time.time() - start_time
        print('time: %f' % (duration))
        print('time prediction needed: ', duration * (np.prod(x_shape)-1-i))
    jacobian = jacobian.reshape(func_shape+x_shape)
    return jacobian

In [ ]:
# check the graident and jacobian
grad = np.zeros(q_init.shape).flatten()
solver.objective_trajopt_f(q_init, grad)
grad_fd = naive_finite_diff(lambda qs: solver.objective_trajopt_f(qs, np.array([])), q_init.flatten())
print('error: ', np.linalg.norm(grad - grad_fd))

In [ ]:
# collision constraint

res = np.zeros(solver.constr_safety_margin.shape)
jac = np.zeros((solver.constr_safety_margin.size, solver.n))
solver.collision_constraint_nlopt_c(res, q_init.flatten(), jac)
def collision_constraint(qs):
    res = np.zeros(solver.constr_safety_margin.shape)
    solver.collision_constraint_nlopt_c(res, qs, np.array([]))
    return res
jac_fd = naive_finite_diff_jacobian(collision_constraint, q_init.flatten())
print('error: ', np.linalg.norm(jac - jac_fd))

In [ ]:
# collision constraint

res = np.zeros(solver.constr_max_dist.shape)
jac = np.zeros((solver.constr_max_dist.size, solver.n))
solver.position_distance_constraint_nlopt_c(res, q_init.flatten(), jac)
def waypoint_distance_constraint(qs):
    res = np.zeros(solver.constr_max_dist.shape)
    solver.position_distance_constraint_nlopt_c(res, qs, np.array([]))
    return res
jac_fd = naive_finite_diff_jacobian(waypoint_distance_constraint, q_init.flatten())
print('error: ', np.linalg.norm(jac - jac_fd))